In [ ]:
base_code='../Documents/codes/Numerical_Methods/Scripts/Burgers_MoC.jl'

generated_code={
'1':'../Results/NM_codes/Burgers_Periodic_SinSum/2025_12_11_2256',
'2':'../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_11_2154',
'3':'../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_15_2212',
}

In [ ]:
folders = {
    '1': '../Results/NM_codes/Burgers_Periodic_SinSum/2025_12_11_2256',
    '2': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_11_2154',
    '3': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_15_2212',
    '4': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_15_2228',
    '5':'../Results/NM_codes/Burgers_Periodic_SinSum/2025_12_15_2248',
    '6':'../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_15_2303',
    '7':'../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1053',
    '8':'../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1233',
    '9':'../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222',
}


In [ ]:
import os
import glob
import re
import difflib
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# --- 1. Formatting Setup ---
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = 12

### Path Configuration
base_path='../Documents/codes/SciML/PINNs/MLP_SSBroyden.py'

folders = {
    'H1': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'H2': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1A2/2025_12_07_1232/Experiment_Report.md',
    'H3': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1331/Experiment_Report.md',
    'H4': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1656/Experiment_Report.md',
    'H5': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1709/Experiment_Report.md',
}

def clean_code_to_lines(code):
    """Removes comments/docstrings and returns a list of stripped lines."""
    # Remove multi-line comments/docstrings (Python style)
    code = re.sub(r'\"\"\"(.*?)\"\"\"', '', code, flags=re.DOTALL)
    code = re.sub(r'\'\'\'(.*?)\'\'\'', '', code, flags=re.DOTALL)
    # Remove single-line comments
    code = re.sub(r'#.*', '', code)
    # Return list of non-empty stripped lines
    return [line.strip() for line in code.splitlines() if line.strip()]

def get_file_content(path):
    """Locates the source code file in the directory of the .md report."""
    if os.path.isfile(path) and (path.endswith('.py') or path.endswith('.jl')):
        target = path
    else:
        # Get the directory from the .md report path
        directory = os.path.dirname(path)
        # Prioritize .py for PINN cases, then .jl
        found = glob.glob(os.path.join(directory, '*.py')) + glob.glob(os.path.join(directory, '*.jl'))
        if not found:
            raise FileNotFoundError(f"No code file found in {directory}")
        target = found[0]
        
    with open(target, 'r', encoding='utf-8') as f:
        return clean_code_to_lines(f.read())

### 1. Data Ingestion
# The keys in 'keys' list will be [0, 'H 1', 'H 2', 'H 3', 'H 4', 'H 5']
codes_lines = {'0': get_file_content(base_path)}
for key, report_path in folders.items():
    codes_lines[key] = get_file_content(report_path)

### 2. Symmetric Matrix Computation (Line-Based)
keys = ['0'] + sorted([k for k in codes_lines.keys() if k != '0'])
n = len(keys)
similarity_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(i, n):
        if i == j:
            similarity_matrix[i, j] = 1.0
        else:
            line_list_a = codes_lines[keys[i]]
            line_list_b = codes_lines[keys[j]]
            
            # Using SequenceMatcher on list of lines (not characters)
            ratio = difflib.SequenceMatcher(None, line_list_a, line_list_b).ratio()
            similarity_matrix[i, j] = ratio
            similarity_matrix[j, i] = ratio

### 3. Visualization
plt.figure(figsize=(7, 6))
# Setting axis labels using your keys specifically
x_labels = [f"{k}" if k != '0' else "Base" for k in keys]
y_labels = x_labels

sns.heatmap(similarity_matrix, annot=True, cmap='RdBu_r',
            xticklabels=x_labels, yticklabels=y_labels, vmin=0, vmax=1)

plt.title("Helmholtz (PINN)")
plt.tight_layout()
plt.savefig('pinn_similarity_map.png')
plt.show()

In [ ]:
import os
import glob
import re
import difflib
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import gridspec

# --- 1. Formatting Setup ---
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = 14

# ==========================================
# Configuration Data
# ==========================================
plot_configs = [
    {
        'title': "Inviscid 2D Burgers (Exact)",
        'base': '../Documents/codes/Numerical_Methods/Scripts/Burgers_MoC.jl',
        'folders': {
            'IB1': '../Results/NM_codes/Burgers_Periodic_SinSum/2025_12_15_2248/',
            'IB2': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_15_2303/',
            'IB3': '../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1053/',
            'IB4': '../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1233/',
            'IB5': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222/',
        }
    },
    {
        'title': "Rayleigh Taylor (Numerical)",
        'base': '../Documents/codes/Numerical_Methods/Scripts/cNS_DGSEM.jl',
        'folders': {
            'RT1': '../Results/NM_codes/CompressibleEuler_PeriodicReflective_SmoothedInterface/2025_12_09_2125/',
            'RT2': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface/2025_12_05_2322/',
            'RT3': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_1059/',
            'RT4': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/',
            'RT5': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/',
        }
    },
    {
        'title': "Helmholtz (PINN)",
        'base': '../Documents/codes/SciML/PINNs/MLP_SSBroyden.py',
        'folders': {
            'H1': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
            'H2': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1A2/2025_12_07_1232/Experiment_Report.md',
            'H3': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1331/Experiment_Report.md',
            'H4': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1656/Experiment_Report.md',
            'H5': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1709/Experiment_Report.md',
        }
    }
]

# ==========================================
# Helper Functions
# ==========================================
def clean_code_to_lines(code_str, ext):
    code_str = code_str.replace('\r\n', '\n')
    if ext == '.jl':
        code_str = re.sub(r'#=.*?=#', '', code_str, flags=re.DOTALL)
    elif ext == '.py':
        code_str = re.sub(r'\"\"\"(.*?)\"\"\"', '', code_str, flags=re.DOTALL)
        code_str = re.sub(r'\'\'\'(.*?)\'\'\'', '', code_str, flags=re.DOTALL)
    clean_lines = []
    for line in code_str.splitlines():
        line = re.sub(r'#.*', '', line)
        stripped = line.strip()
        if stripped: clean_lines.append(stripped)
    return clean_lines

def get_file_content(path_in):
    target_file = None
    if os.path.isfile(path_in):
        if path_in.endswith('.md'):
            directory = os.path.dirname(path_in)
            found = glob.glob(os.path.join(directory, '*.py')) + glob.glob(os.path.join(directory, '*.jl'))
            if found: target_file = found[0]
        else: target_file = path_in
    elif os.path.isdir(path_in):
        found = glob.glob(os.path.join(path_in, '*.jl')) + glob.glob(os.path.join(path_in, '*.py'))
        if found: target_file = found[0]
    if not target_file or not os.path.exists(target_file): return []
    _, ext = os.path.splitext(target_file)
    try:
        with open(target_file, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        return clean_code_to_lines(content, ext)
    except: return []

# ==========================================
# Main Execution & Precise Layout Plotting
# ==========================================
# 1. Figure setup (12x5)
fig = plt.figure(figsize=(12, 5))

# 2. GridSpec width_ratios: [1, 1, 1, 0.05] creates space for the colorbar
gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, 0.15], wspace=0.1)

for idx, config in enumerate(plot_configs):
    ax = fig.add_subplot(gs[idx])
    
    # Data Processing
    codes_map = {'0': get_file_content(config['base'])}
    for key, path in config['folders'].items():
        codes_map[key] = get_file_content(path)
    
    keys = ['0'] + sorted([k for k in codes_map.keys() if k != '0'])
    n = len(keys)
    similarity_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(i, n):
            lines_a, lines_b = codes_map[keys[i]], codes_map[keys[j]]
            ratio = difflib.SequenceMatcher(None, lines_a, lines_b).ratio() if lines_a and lines_b else 0.0
            similarity_matrix[i, j] = similarity_matrix[j, i] = ratio

    labels = ["Base"] + [re.search(r'\d+', k).group() for k in keys if k != '0']
    
    # Plotting: square=True maintains 1:1 aspect ratio for cells
    sns.heatmap(similarity_matrix, annot=True, fmt=".2f", cmap='RdBu_r', 
                vmin=0, vmax=1, ax=ax, square=True, annot_kws={"size": 12},
                xticklabels=labels, yticklabels=labels if idx == 0 else False, 
                cbar=False)
    
    ax.set_title(config['title'], fontsize=12, pad=12)
    ax.set_xlabel("Run", fontsize=12)
    if idx == 0:
        ax.set_ylabel("Run", fontsize=12)

# --- 3. Add Single Colorbar With Controlled Height ---
# Create an axis for the colorbar specifically in the 4th GridSpec slot
cbar_ax = fig.add_subplot(gs[3])

# Create the colorbar using a ScalarMappable
norm = mpl.colors.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=norm)
sm.set_array([])

# We use fraction=0.04 and pad=0 to keep it thin. 
# We adjust its size relative to cbar_ax using anchor/shrink if needed
cb = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')

# REDUCE HEIGHT: Force the colorbar axis to be shorter so it matches the heatmaps
# In a 12x5 figure with square heatmaps, the heatmaps take ~75% of the height
pos = cbar_ax.get_position()
cbar_ax.set_position([pos.x0, pos.y0 + pos.height * 0.12, pos.width * 0.3, pos.height * 0.76])

plt.savefig('combined_similarity_v5.png', dpi=300, bbox_inches='tight')
plt.show()

## Logs

In [ ]:
import os
import glob
import re

### Path Configuration
folders = {
    '1':'../Results/NM_codes/Burgers_Periodic_SinSum/2025_12_15_2248',
    '2':'../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_15_2303',
    '3':'../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1053',
    '4':'../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1233',
    '5':'../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222',
}

def parse_full_history_data(md_text):
    """
    Identifies the 'Full Trial History' section and extracts 
    the iteration, full description, and score from the last row.
    """
    # 1. Capture the block between 'Full Trial History' and the next Header
    section_pattern = r'## Full Trial History\s*(.*?)(?=\n##|$)'
    match = re.search(section_pattern, md_text, re.DOTALL | re.IGNORECASE)
    
    if not match:
        return None, None, None
    
    section_content = match.group(1)
    
    # 2. Extract lines formatted as Markdown table rows
    rows = re.findall(r'^\|(.*)\|', section_content, re.MULTILINE)
    
    # 3. Filter out non-data rows (Header and Separator)
    data_rows = []
    for row in rows:
        if "Run" in row or re.match(r'^[\s:|:-]*$', row) or not row.strip():
            continue
        data_rows.append(row)
    
    if not data_rows:
        return None, None, None
    
    # 4. Extract columns from the LAST row
    last_row = data_rows[-1]
    cols = [c.strip() for c in last_row.split('|')]
    
    # Column Index Mapping:
    # [0] Run (Iteration) | [1] Description | [-1] Total Score
    iteration = cols[0]
    full_description = cols[1]
    total_score = cols[-1].replace('**', '') # Clean bolding
    
    return iteration, full_description, total_score

### Dictionaries to store the results
iterations = {}
descriptions = {}
scores = {}

for case_id, path in folders.items():
    # Look for the .md report file
    md_files = glob.glob(os.path.join(path, "*.md"))
    if not md_files:
        print(f"!!! Error: No .md file found in Case {case_id}")
        continue
        
    with open(md_files[0], 'r', encoding='utf-8') as f:
        content = f.read()
    
    it, desc, sc = parse_full_history_data(content)
    
    if it:
        iterations[case_id] = it
        descriptions[case_id] = desc
        scores[case_id] = sc


### Full Extraction Results

# Iterate through sorted keys and print everything without truncation
for case_id in sorted(scores.keys()):
    print(f"============================================================")
    print(f"CASE ID: {case_id}")
    print(f"============================================================")
    print(f"LAST ITERATION (RUN): {iterations[case_id]}")
    print(f"TOTAL SCORE:          {scores[case_id]}")
    print(f"------------------------------------------------------------")
    print(f"FULL MODEL DESCRIPTION:")
    print(f"{descriptions[case_id]}")
    print(f"------------------------------------------------------------\n")

In [ ]:
import os
import glob
import re

### Path Configuration
folders = {
            'RT1': '../Results/NM_codes/CompressibleEuler_PeriodicReflective_SmoothedInterface/2025_12_09_2125/',
            'RT2': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface/2025_12_05_2322/',
            'RT3': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_1059/',
            'RT4': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/',
            'RT5': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/',
}

def parse_full_history_data(md_text):
    """
    Identifies the 'Full Trial History' section and extracts 
    the iteration, full description, and score from the last row.
    """
    # 1. Capture the block between 'Full Trial History' and the next Header
    section_pattern = r'## Full Trial History\s*(.*?)(?=\n##|$)'
    match = re.search(section_pattern, md_text, re.DOTALL | re.IGNORECASE)
    
    if not match:
        return None, None, None
    
    section_content = match.group(1)
    
    # 2. Extract lines formatted as Markdown table rows
    rows = re.findall(r'^\|(.*)\|', section_content, re.MULTILINE)
    
    # 3. Filter out non-data rows (Header and Separator)
    data_rows = []
    for row in rows:
        if "Run" in row or re.match(r'^[\s:|:-]*$', row) or not row.strip():
            continue
        data_rows.append(row)
    
    if not data_rows:
        return None, None, None
    
    # 4. Extract columns from the LAST row
    last_row = data_rows[-1]
    cols = [c.strip() for c in last_row.split('|')]
    
    # Column Index Mapping:
    # [0] Run (Iteration) | [1] Description | [-1] Total Score
    iteration = cols[0]
    full_description = cols[1]
    total_score = cols[-1].replace('**', '') # Clean bolding
    
    return iteration, full_description, total_score

### Dictionaries to store the results
iterations = {}
descriptions = {}
scores = {}

for case_id, path in folders.items():
    # Look for the .md report file
    md_files = glob.glob(os.path.join(path, "*.md"))
    if not md_files:
        print(f"!!! Error: No .md file found in Case {case_id}")
        continue
        
    with open(md_files[0], 'r', encoding='utf-8') as f:
        content = f.read()
    
    it, desc, sc = parse_full_history_data(content)
    
    if it:
        iterations[case_id] = it
        descriptions[case_id] = desc
        scores[case_id] = sc


### Full Extraction Results

# Iterate through sorted keys and print everything without truncation
for case_id in sorted(scores.keys()):
    print(f"============================================================")
    print(f"CASE ID: {case_id}")
    print(f"============================================================")
    print(f"LAST ITERATION (RUN): {iterations[case_id]}")
    print(f"TOTAL SCORE:          {scores[case_id]}")
    print(f"------------------------------------------------------------")
    print(f"FULL MODEL DESCRIPTION:")
    print(f"{descriptions[case_id]}")
    print(f"------------------------------------------------------------\n")

In [ ]:
import os
import glob
import re

### Path Configuration
folders = {
            'H1': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/',
            'H2': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1A2/2025_12_07_1232/',
            'H3': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1331/',
            'H4': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1656/',
            'H5': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1709/',
}

def parse_full_history_data(md_text):
    """
    Identifies the 'Full Trial History' section and extracts 
    the iteration, full description, and score from the last row.
    """
    # 1. Capture the block between 'Full Trial History' and the next Header
    section_pattern = r'## Full Trial History\s*(.*?)(?=\n##|$)'
    match = re.search(section_pattern, md_text, re.DOTALL | re.IGNORECASE)
    
    if not match:
        return None, None, None
    
    section_content = match.group(1)
    
    # 2. Extract lines formatted as Markdown table rows
    rows = re.findall(r'^\|(.*)\|', section_content, re.MULTILINE)
    
    # 3. Filter out non-data rows (Header and Separator)
    data_rows = []
    for row in rows:
        if "Run" in row or re.match(r'^[\s:|:-]*$', row) or not row.strip():
            continue
        data_rows.append(row)
    
    if not data_rows:
        return None, None, None
    
    # 4. Extract columns from the LAST row
    last_row = data_rows[-1]
    cols = [c.strip() for c in last_row.split('|')]
    
    # Column Index Mapping:
    # [0] Run (Iteration) | [1] Description | [-1] Total Score
    iteration = cols[0]
    full_description = cols[1]
    total_score = cols[-1].replace('**', '') # Clean bolding
    
    return iteration, full_description, total_score

### Dictionaries to store the results
iterations = {}
descriptions = {}
scores = {}

for case_id, path in folders.items():
    # Look for the .md report file
    md_files = glob.glob(os.path.join(path, "*.md"))
    if not md_files:
        print(f"!!! Error: No .md file found in Case {case_id}")
        continue
        
    with open(md_files[0], 'r', encoding='utf-8') as f:
        content = f.read()
    
    it, desc, sc = parse_full_history_data(content)
    
    if it:
        iterations[case_id] = it
        descriptions[case_id] = desc
        scores[case_id] = sc


### Full Extraction Results

# Iterate through sorted keys and print everything without truncation
for case_id in sorted(scores.keys()):
    print(f"============================================================")
    print(f"CASE ID: {case_id}")
    print(f"============================================================")
    print(f"LAST ITERATION (RUN): {iterations[case_id]}")
    print(f"TOTAL SCORE:          {scores[case_id]}")
    print(f"------------------------------------------------------------")
    print(f"FULL MODEL DESCRIPTION:")
    print(f"{descriptions[case_id]}")
    print(f"------------------------------------------------------------\n")